# Análisis de campañas de navegación semántica en ROS 2

Analiza campañas ya generadas, verifica la configuración congelada y mantiene separados casos, runs, campañas y escenas. No ajusta pesos ni umbrales.


In [ ]:
from pathlib import Path
import sys

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_navigation_ws" / "src").is_dir())
sys.path.insert(0, str(repo / "experiments" / "shared"))

import pandas as pd

from notebook_bootstrap import bootstrap_simulation, find_repo_root

simulation_ready = True
try:
    ctx = bootstrap_simulation()
except (FileNotFoundError, ValueError, RuntimeError) as error:
    simulation_ready = False
    ctx = {"repo_root": find_repo_root(), "bootstrap_error": str(error)}
    print("Análisis bloqueado hasta disponer de configuración congelada:", error)


## Descubrimiento y validación de campañas


In [ ]:
from semantic_evaluation.core.campaign_analysis import (
    aggregate_campaign_levels, discover_runs, load_campaign_cases)
from semantic_evaluation.core.config_validation import expand_path

runs = []
cases = pd.DataFrame()
campaign_issues = []
if simulation_ready:
    campaigns_root, missing = expand_path(
        ctx["config"]["paths"]["campaigns_root"], ctx["repo_root"])
    runs = discover_runs(campaigns_root)
    cases, campaign_issues = load_campaign_cases(runs, ctx["frozen_config_hash"])
print({"runs_discovered": len(runs), "cases_valid": len(cases),
       "issues": campaign_issues})
if simulation_ready and not runs:
    print("Faltan campaigns/<scene_id>/<run_id>/{evaluation.csv,manifest.json,campaign.yaml}.")
    print("Genéralos con simulation.launch.py y evaluation.launch.py según docs/EXPERIMENTS.md.")


## Caso, run, campaña y escena


In [ ]:
summaries = aggregate_campaign_levels(
    cases,
    confidence_level=ctx["config"]["campaign_analysis"]["confidence_level"] if simulation_ready else 0.95,
    bootstrap_samples=ctx["config"]["campaign_analysis"]["bootstrap_samples"] if simulation_ready else 2000,
) if not cases.empty else {"runs": pd.DataFrame(), "campaigns": pd.DataFrame(), "scenes": pd.DataFrame()}
display(cases.head())
display(summaries["runs"])
display(summaries["campaigns"])
display(summaries["scenes"])


## Métricas semánticas, navegación y fallos


In [ ]:
if not cases.empty:
    display(cases.groupby(["scene_id", "method", "query_type", "language"], dropna=False)[
        ["recall_at_1", "recall_at_3", "recall_at_5", "reciprocal_rank",
         "semantic_success", "nearby_semantic_success", "navigation_success",
         "end_to_end_success", "retrieval_latency_ms", "navigation_time_s"]
    ].mean(numeric_only=True))
    display(cases.groupby(["scene_id", "method", "failure_type"]).size()
            .rename("cases").reset_index())
else:
    print("No hay casos; no se calculan métricas ni conclusiones.")


## Calidad de grafos por escena


In [ ]:
from dataclasses import asdict
from semantic_evaluation.core.config_validation import expand_path
from semantic_evaluation.core.graph_analysis import safe_analyze_graph

graph_rows = []
graph_issues = []
if simulation_ready:
    for scene in ctx["scenes"]:
        if not scene.enabled:
            continue
        path, missing = expand_path(scene.graph_db, ctx["repo_root"])
        if missing or path is None:
            graph_issues.append({"scene_id": scene.scene_id, "issue": "missing_environment",
                                 "detail": missing})
            continue
        quality, issues = safe_analyze_graph(scene.scene_id, path)
        graph_rows.append(asdict(quality))
        graph_issues.extend(issues)
display(pd.DataFrame(graph_rows))
display(pd.DataFrame(graph_issues))


## Visualizaciones


In [ ]:
import matplotlib.pyplot as plt
from plotting import METHOD_LABELS, apply_plot_style, save_figure

apply_plot_style()
if not cases.empty:
    figures_root, _ = expand_path(ctx["config"]["paths"]["results_root"], ctx["repo_root"])
    figures_root = figures_root / "figures"
    for metric, title, name in [
        ("semantic_success", "Éxito semántico por escena", "semantic_success_by_scene"),
        ("navigation_success", "Éxito de navegación por escena", "navigation_success_by_scene"),
        ("end_to_end_success", "Éxito extremo a extremo por escena", "end_to_end_success_by_scene"),
    ]:
        table = cases.groupby(["scene_id", "method"])[metric].mean().unstack()
        axis = table.rename(columns=METHOD_LABELS).plot.bar(figsize=(9, 4), ylim=(0, 1), title=title)
        axis.set_ylabel("proporción")
        save_figure(axis.figure, figures_root, name)
        plt.show()


## Exportación e interpretación


In [ ]:
from pathlib import Path
from semantic_evaluation.core.campaign_analysis import export_campaign_analysis

if simulation_ready and not cases.empty:
    results_root, _ = expand_path(ctx["config"]["paths"]["results_root"], ctx["repo_root"])
    export_campaign_analysis(results_root, cases, summaries, campaign_issues)
    graphs_dir = results_root / "graphs"
    graphs_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(graph_rows).to_parquet(graphs_dir / "graph_quality.parquet", index=False)
    pd.DataFrame(graph_issues).to_parquet(graphs_dir / "graph_issues.parquet", index=False)
    best = summaries["scenes"].dropna(subset=["end_to_end_success"]).sort_values(
        "end_to_end_success", ascending=False)
    if best.empty:
        print("No hay suficientes valores para interpretar el éxito extremo a extremo.")
    else:
        row = best.iloc[0]
        print(f"El mayor éxito extremo a extremo observado fue {row['end_to_end_success']:.3f} "
              f"para {row['method']} en {row['scene_id']} (n={row['n_queries']}).")
else:
    print("No se exportan resultados hasta disponer de configuración compatible y campañas válidas.")
